In [7]:
import os
import pandas as pd
import json
import numpy as np

In [8]:
# Define the expected columns
expected_columns = {
    'totalSimSamples',
    'simId',
    "Scene", "Scenario", "Purpose", "EngineScript_sampleInterval",
    "TaskScript_start", "TaskScript_end", "TaskScript_pointsOfInterest",
    "TaskScript_numberOfAgents", "TaskScript_spawnInterval", "TaskScript_taskName",
    "TaskScript_agentType", "TaskScript_numberOfNeeds", "TaskScript_agentSpeed",
    "TaskScript_agentSize", "TaskScript_agentRadius", "TaskScript_privacyRadius",
    "TaskScript_poiTime", "TaskScript_revisit", "TaskScript_chooseNonDeterministically"
}

In [9]:
# Directory containing the CSV files
#csv_dir = "Simulation setup 20241114"
#csv_dir = "/home/leo/ETH/Michal/Jiayu/experiment/Simulation setup 202502/"
#output_base_dir = "output_json"

csv_dir = "Simulation setup 202503"
output_base_dir = "output_json_Test_2"
os.makedirs(output_base_dir, exist_ok=True)

In [10]:
class CustomEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer, np.floating)):
            return obj.item()  # Converts to native Python int/float
        if isinstance(obj, np.ndarray):
            return obj.tolist()  # Converts numpy arrays to lists
        return super().default(obj)

In [11]:
def all_sim_values_same(df, columns=["Scenario", "Purpose", "EngineScript_sampleInterval"]):
    return (df[columns].nunique() == 1).all()

In [12]:
# Process each CSV file
for filename in os.listdir(csv_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(csv_dir, filename)
        df = pd.read_csv(filepath)
        
        # Fill optional columns if not present
        if 'totalSimSamples' not in df.columns:
            df['totalSimSamples']=1
            df['totalSimSamples'].astype(int).map(int)
        if 'simId' not in df.columns:
            df.index.name = 'simId'  # Optional, if not already set
            df = df.reset_index()
            df['simId'].astype(int).map(int)
        
        # Check if all required columns are present
        if not expected_columns.issubset(df.columns):
            missing = expected_columns - set(df.columns)
            print(f"Skipping {filename}, missing columns: {missing}")
            continue
                
        # Extract the prefix (e.g., PT1) from the filename
        prefix = filename.split("-")[0].strip()
        output_dir = os.path.join(output_base_dir, prefix)
        os.makedirs(output_dir, exist_ok=True)
        
        common_columns=[]
        task_columns=[]

        for c in expected_columns:
            if 'TaskScript_' not in c:
                common_columns.append(c)
            else:
                task_columns.append(c)

        for simId, sim_df in df.groupby('simId'):
            sim_config={}
            if not all_sim_values_same(sim_df,common_columns):
                print(f"Error in simID {simID}: different values for simulation")
                continue

            for c in common_columns:
                sim_config[c]=sim_df[c].iloc[0]
            # Extract task details
            tasks = []
            for _, task in sim_df.iterrows():
                task_dict = {}
                for tc in task_columns:
                    task_dict[tc.replace('TaskScript_','')]=task[tc]
                # Add more colors in the future
                if task_dict['taskName']=='Infected':
                    task_dict['taskColor']='red'
                else:
                    task_dict['taskColor']='black'
                tasks.append(task_dict)

            sim_config['tasks']=tasks
            num_samples=sim_config['totalSimSamples']
            sim_id=sim_config['simId']
            del sim_config['totalSimSamples']
            
            for idy in range(num_samples):
                sim_config['sampleNum']=idy
                json_filename = os.path.join(output_dir, f"{prefix}_simId_{sim_id}_sampleNum_{idy}.json")
                with open(json_filename, "w") as f:
                    json.dump(sim_config, f, indent=2,cls=CustomEncoder)
print("Processing complete.")

Processing complete.
